# LeetCode 785 — Is Graph Bipartite? bằng Python

Link đề: https://leetcode.com/problems/is-graph-bipartite/ · Độ khó: **Medium** · Chủ đề: Graph (đồ thị), DFS, BFS, Union Find (tập hợp rời nhau), Graph Coloring (tô màu đồ thị), Bipartite Graph (đồ thị hai phía)

Skeleton LeetCode:

```python
class Solution:
    def isBipartite(self, graph: List[List[int]]) -> bool:
        ...
```

Ghi chú: bài hỏi một **tính chất** (có/không) của đồ thị, không có đáp án nhiều dạng; nhưng khi ta muốn **chỉ ra bằng chứng** thì lại có nhiều cách phân hai phần đúng, vì vậy notebook kiểm tra bằng **hàm kiểm tra tính hợp lệ** (một cách tô màu hợp lệ, hoặc một **chu trình lẻ**) bên cạnh oracle vét cạn. Ý tưởng chính: tô màu hai màu bằng DFS/BFS, hoặc dùng Union Find; và định lý "**đồ thị hai phía ⇔ không có chu trình độ dài lẻ**".

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đồ thị vô hướng `n` đỉnh cho bằng **danh sách kề**; hỏi có chia được tập đỉnh thành hai tập `A`, `B` sao cho **mọi cạnh nối một đỉnh `A` với một đỉnh `B`** không.
2. Nắm các công cụ Python cần thiết: danh sách kề, `collections.deque` (hàng đợi cho BFS), đệ quy DFS, mảng màu với giá trị `0` (chưa tô) / `1` / `-1`, vòng lặp bao phủ **mọi thành phần liên thông**, Union Find (tìm cha có nén đường đi).
3. Hiểu định lý: **hai phía ⇔ không có chu trình lẻ**; và vì sao khi tô màu, gặp một cạnh nối hai đỉnh **cùng màu** là gặp một chu trình lẻ.
4. Đi qua 5 cách: thử mọi cách tô `2^n`, DFS tô màu, BFS tô màu, Union Find "mọi láng giềng chung tập", Union Find có **tính chẵn lẻ**; đối chiếu bằng oracle vét cạn và bằng **bằng chứng** (cách tô hợp lệ hoặc chu trình lẻ).
5. Tránh các lỗi kinh điển: chỉ chạy từ đỉnh 0 (đồ thị không liên thông), dùng `visited` thay vì màu, dùng màu `0/1` trùng với "chưa tô", nghĩ rằng "có chu trình là không hai phía", quên kiểm tra cạnh tới đỉnh đã tô.

## 2. Đề bài & ví dụ

Cho đồ thị **vô hướng** có `n` đỉnh đánh số `0..n-1`. Mảng 2 chiều `graph` có `graph[u]` là danh sách các đỉnh kề với `u`. Đồ thị thỏa:

- Không có khuyên (`graph[u]` không chứa `u`).
- Không có cạnh song song (`graph[u]` không có giá trị trùng).
- Nếu `v` thuộc `graph[u]` thì `u` thuộc `graph[v]` (vô hướng).
- Đồ thị **có thể không liên thông**.

Đồ thị là **hai phía** (bipartite) nếu có thể chia các đỉnh thành hai tập **độc lập** `A`, `B` (không có cạnh nối hai đỉnh trong cùng một tập) sao cho **mọi cạnh** nối một đỉnh của `A` với một đỉnh của `B`. Trả về `true` khi và chỉ khi đồ thị là hai phía.

**Ví dụ 1**

- Input: `graph = [[1,2,3],[0,2],[0,1,3],[0,2]]`
- Output: `false`
- Giải thích: các cạnh `0-1`, `1-2`, `0-2` tạo thành **tam giác** (chu trình độ dài 3, lẻ) nên không thể chia hai tập.

**Ví dụ 2**

- Input: `graph = [[1,3],[0,2],[1,3],[0,2]]`
- Output: `true`
- Giải thích: chia `{0, 2}` và `{1, 3}`: mọi cạnh nối một đỉnh mỗi tập.

**Ràng buộc:**

- `graph.length == n`, `1 <= n <= 100`
- `0 <= graph[u].length < n`, `0 <= graph[u][i] <= n - 1`
- `graph[u]` không chứa `u`; các giá trị của `graph[u]` đôi một khác nhau.
- Nếu `graph[u]` chứa `v` thì `graph[v]` chứa `u`.

In [1]:
from typing import List, Optional, Tuple
from collections import deque
from itertools import product
import random, time

examples = [
    ([[1, 2, 3], [0, 2], [0, 1, 3], [0, 2]], False),
    ([[1, 3], [0, 2], [1, 3], [0, 2]], True),
]
for g, expected in examples:
    edges = sum(len(a) for a in g) // 2
    print(f"n={len(g)} số cạnh={edges} graph={g} kỳ vọng={expected}")

n=4 số cạnh=5 graph=[[1, 2, 3], [0, 2], [0, 1, 3], [0, 2]] kỳ vọng=False
n=4 số cạnh=4 graph=[[1, 3], [0, 2], [1, 3], [0, 2]] kỳ vọng=True


## 3. Kiến thức nền

### 3.1. Danh sách kề

`graph[u]` liệt kê các đỉnh kề với `u`. Vì đồ thị vô hướng, mỗi cạnh `u - v` xuất hiện **hai lần** (`v` trong `graph[u]` và `u` trong `graph[v]`), nên số cạnh = tổng độ dài các danh sách / 2. Ta sẽ cần hàm dựng danh sách kề từ danh sách cạnh để tự tạo test.

In [2]:
def build_graph(n: int, edges: List[Tuple[int, int]]) -> List[List[int]]:
    graph = [[] for _ in range(n)]
    for u, v in edges:
        graph[u].append(v)
        graph[v].append(u)
    return graph

def edge_list(graph: List[List[int]]) -> List[Tuple[int, int]]:
    return [(u, v) for u in range(len(graph)) for v in graph[u] if u < v]

g = build_graph(4, [(0, 1), (1, 2), (2, 3), (3, 0)])
print(g, "-> cạnh:", edge_list(g))
assert sorted(edge_list(examples[1][0])) == [(0, 1), (0, 3), (1, 2), (2, 3)]
print("tam giác:", build_graph(3, [(0, 1), (1, 2), (2, 0)]))

[[1, 3], [0, 2], [1, 3], [2, 0]] -> cạnh: [(0, 1), (0, 3), (1, 2), (2, 3)]
tam giác: [[1, 2], [0, 2], [1, 0]]


### 3.2. Tô màu hai màu và quy tắc cạnh

Gán mỗi đỉnh một trong hai màu (ta dùng `1` và `-1`, còn `0` nghĩa là **chưa tô**). Hợp lệ khi **mọi cạnh nối hai đỉnh khác màu**. Hai tập `A` (màu `1`), `B` (màu `-1`) chính là hai phía cần tìm. Dùng `-c` để đổi sang màu còn lại.

In [3]:
def valid_coloring(graph, color) -> bool:
    """color[u] ∈ {1, -1}; hợp lệ khi mọi cạnh nối hai đỉnh khác màu."""
    return all(color[u] in (1, -1) for u in range(len(graph))) and \
           all(color[u] != color[v] for u in range(len(graph)) for v in graph[u])

g = [[1, 3], [0, 2], [1, 3], [0, 2]]
print(valid_coloring(g, [1, -1, 1, -1]), valid_coloring(g, [1, 1, -1, -1]))
c = 1
print("đổi màu:", -c, -(-c))

True False
đổi màu: -1 1


### 3.3. `deque` cho BFS

BFS (duyệt theo chiều rộng) cần hàng đợi lấy ra từ đầu, thêm vào cuối. `collections.deque` làm cả hai thao tác trong `O(1)` (list Python `pop(0)` thì `O(n)`).

In [4]:
dq = deque([5])
dq.append(7); dq.append(9)
print(dq, dq.popleft(), dq)

deque([7, 9]) 5 deque([7, 9])


### 3.4. Đồ thị không liên thông: duyệt từ mọi đỉnh chưa thăm

Một lần DFS/BFS chỉ phủ **một thành phần liên thông**. Đồ thị có thể gồm nhiều thành phần nên cần vòng lặp ngoài: với mọi đỉnh `i`, nếu chưa tô thì bắt đầu một lần duyệt mới từ `i`. (Mỗi thành phần được tô độc lập, nên màu bắt đầu có thể chọn tùy ý.)

In [5]:
def components(graph):
    seen = [False] * len(graph)
    comps = []
    for s in range(len(graph)):
        if seen[s]:
            continue
        comp, stack = [], [s]
        seen[s] = True
        while stack:
            u = stack.pop()
            comp.append(u)
            for v in graph[u]:
                if not seen[v]:
                    seen[v] = True
                    stack.append(v)
        comps.append(sorted(comp))
    return comps

g = build_graph(7, [(0, 1), (1, 2), (3, 4), (5, 5 + 1)])
print(components(g))

[[0, 1, 2], [3, 4], [5, 6]]


### 3.5. Union Find (DSU) và nén đường đi

Union Find quản lý các tập rời nhau: `find(x)` trả đại diện (gốc) của tập chứa `x`; `union(a, b)` gộp hai tập. **Nén đường đi** (`p[x] = find(p[x])`) làm các lần `find` sau gần như `O(1)`.

In [6]:
p = list(range(6))
def find(x):
    if p[x] != x:
        p[x] = find(p[x])             # nén đường đi
    return p[x]
def union(a, b):
    p[find(a)] = find(b)

union(0, 1); union(1, 2); union(3, 4)
print([find(i) for i in range(6)], "| 0 và 2 cùng tập?", find(0) == find(2), "| 0 và 3?", find(0) == find(3))

[2, 2, 2, 4, 4, 5] | 0 và 2 cùng tập? True | 0 và 3? False


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Hãy tô màu **như chơi cờ**: chọn một đỉnh, tô màu `A`; mọi láng giềng của nó **buộc** phải là `B`; láng giềng của các đỉnh `B` lại buộc là `A`, và cứ thế. Nếu có xung đột (một đỉnh bị ép cả `A` lẫn `B`) thì không thể chia hai phía.

Ví dụ 2 `[[1,3],[0,2],[1,3],[0,2]]` (một chu trình 4): đỉnh `0` -> `A`; `1`, `3` -> `B`; láng giềng của `1` là `0` (A ✓) và `2` -> `A`; láng giềng của `3` là `0` (A ✓) và `2` (A ✓ khớp). Không xung đột: `{0, 2}` và `{1, 3}`.

Ví dụ 1: `0` -> `A`; `1`, `2`, `3` -> `B`; nhưng cạnh `1 - 2` nối hai đỉnh cùng màu `B`: **xung đột** -> `false`. Tam giác `0-1-2` là chu trình độ dài 3 (lẻ): đi quanh chu trình lẻ, màu phải đổi một số **lẻ** lần nên không thể quay về màu cũ.

In [7]:
def color_bfs_from_zero(graph):
    """Tô màu thủ công từ đỉnh 0 (chỉ thành phần của 0), trả (color, conflict_edge)."""
    color = [0] * len(graph)
    color[0] = 1
    q = deque([0])
    while q:
        u = q.popleft()
        for v in graph[u]:
            if color[v] == 0:
                color[v] = -color[u]
                q.append(v)
            elif color[v] == color[u]:
                return color, (u, v)
    return color, None

for g, expected in examples:
    color, conflict = color_bfs_from_zero(g)
    print("màu:", color, "| cạnh xung đột:", conflict, "-> hai phía?", conflict is None)
    assert (conflict is None) == expected

màu: [1, -1, -1, -1] | cạnh xung đột: (1, 2) -> hai phía? False
màu: [1, -1, 1, -1] | cạnh xung đột: None -> hai phía? True


## 5. Cách 1 — Brute Force: thử mọi cách tô `2^n`

**Ý tưởng.** Mỗi đỉnh có 2 màu, nên có `2^n` cách tô. Với mỗi cách, kiểm tra **mọi cạnh** nối hai đỉnh khác màu. Nếu có cách nào đạt thì hai phía.

Đây là định nghĩa trực tiếp nên cũng là **oracle** (chỉ dùng cho `n` nhỏ).

In [8]:
def bip_brute(graph: List[List[int]]) -> bool:
    n = len(graph)
    edges = edge_list(graph)
    for colors in product((1, -1), repeat=n):
        if all(colors[u] != colors[v] for u, v in edges):
            return True
    return False

for g, expected in examples:
    got = bip_brute(g)
    print(g, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[[1, 2, 3], [0, 2], [0, 1, 3], [0, 2]] -> False OK
[[1, 3], [0, 2], [1, 3], [0, 2]] -> True OK


**Độ phức tạp và nút thắt.** `O(2^n * E)`: với `n = 100` là `2^100` — không thể. Nút thắt: mỗi lần tô một đỉnh, màu của láng giềng đã **bị ép** (chỉ có 1 lựa chọn trong mỗi thành phần liên thông, sau khi chọn màu một đỉnh), nên không cần thử cả hai màu ở mọi đỉnh.

In [9]:
print(f"{'n':>4} {'số cách tô 2^n':>22}")
for n in (10, 20, 30, 50, 100):
    print(f"{n:>4} {2 ** n:>22,}")

   n         số cách tô 2^n
  10                  1,024
  20              1,048,576
  30          1,073,741,824
  50  1,125,899,906,842,624
 100 1,267,650,600,228,229,401,496,703,205,376


## 6. Ý tưởng thuật toán / bước đột phá

**Màu bị ép.** Trong **một thành phần liên thông**, chọn màu cho một đỉnh thì màu của mọi đỉnh còn lại **hoàn toàn xác định**: láng giềng đổi màu, láng giềng của láng giềng đổi lại... Vậy mỗi thành phần chỉ có tối đa **2 cách tô** (hai cách là hoán đổi màu). Ta chỉ cần **một lần duyệt** mỗi thành phần: tô theo luật "láng giềng khác màu", và phát hiện **xung đột** (cạnh nối hai đỉnh cùng màu).

Thêm một cách nhìn: **định lý** — đồ thị là hai phía ⇔ **không có chu trình độ dài lẻ**. Nếu BFS/DFS gặp cạnh nối hai đỉnh cùng màu thì ta dựng được một chu trình lẻ cụ thể (mục 10).

Các cách triển khai:

| Cách | Ý tưởng |
|---|---|
| 2 | DFS đệ quy: tô `c` cho đỉnh, đệ quy `-c` cho láng giềng chưa tô, xung đột nếu láng giềng đã tô `c` |
| 3 | BFS bằng hàng đợi: cùng luật, duyệt theo tầng |
| 4 | Union Find: mọi láng giềng của một đỉnh phải **cùng tập** với nhau và **khác tập** với đỉnh đó |
| 5 | Union Find có **tính chẵn lẻ** (độ lệch màu so với gốc): mỗi cạnh ép hai đỉnh khác màu |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DFS tô màu

**Giải thích từng đoạn.**

- `color[u]`: `0` chưa tô, `1` hoặc `-1` là màu.
- `dfs(u, c)`: tô `u` màu `c`; với mỗi láng giềng `v`: nếu `color[v] == c` -> **xung đột** (`False`); nếu `color[v] == 0` -> đệ quy `dfs(v, -c)` (nếu nó báo xung đột thì `False`); nếu `color[v] == -c` thì đã đúng, bỏ qua.
- Vòng ngoài duyệt **mọi** đỉnh `i` chưa tô để phủ mọi thành phần.

In [10]:
def bip_dfs(graph: List[List[int]]) -> bool:
    n = len(graph)
    color = [0] * n

    def dfs(u: int, c: int) -> bool:
        color[u] = c
        for v in graph[u]:
            if color[v] == c:
                return False
            if color[v] == 0 and not dfs(v, -c):
                return False
        return True

    for i in range(n):
        if color[i] == 0 and not dfs(i, 1):
            return False
    return True

for g, expected in examples:
    assert bip_dfs(g) == expected
print("DFS OK")

DFS OK


**Code nộp LeetCode (Cách 2):**

In [11]:
class Solution:
    def isBipartite(self, graph: List[List[int]]) -> bool:
        def dfs(a: int, c: int) -> bool:
            color[a] = c
            for b in graph[a]:
                if color[b] == c or (color[b] == 0 and not dfs(b, -c)):
                    return False
            return True

        n = len(graph)
        color = [0] * n
        for i in range(n):
            if color[i] == 0 and not dfs(i, 1):
                return False
        return True

sol_dfs = Solution().isBipartite
print([sol_dfs(g) for g, _ in examples])

[False, True]


**Độ phức tạp (Cách 2):** thời gian `O(n + E)` (mỗi đỉnh một lần, mỗi cạnh xét hai lần), bộ nhớ `O(n)` (mảng màu + ngăn xếp đệ quy tối đa `n <= 100` tầng, an toàn).

---

### 7.2. Cách 3 — BFS tô màu

**Giải thích từng đoạn.**

- Cùng luật tô màu, nhưng dùng hàng đợi: lấy `u` ra, với mỗi láng giềng `v`: chưa tô -> tô `-color[u]` rồi cho vào hàng đợi; đã tô cùng màu với `u` -> xung đột.
- **Tô màu ngay khi đưa vào hàng đợi** (không đợi lúc lấy ra) để một đỉnh không bị đưa vào hai lần với màu khác nhau.
- Vòng ngoài vẫn phủ mọi thành phần.

In [12]:
def bip_bfs(graph: List[List[int]]) -> bool:
    n = len(graph)
    color = [0] * n
    for s in range(n):
        if color[s]:
            continue
        color[s] = 1
        q = deque([s])
        while q:
            u = q.popleft()
            for v in graph[u]:
                if color[v] == 0:
                    color[v] = -color[u]
                    q.append(v)
                elif color[v] == color[u]:
                    return False
    return True

for g, expected in examples:
    assert bip_bfs(g) == expected
print("BFS OK")

BFS OK


**Code nộp LeetCode (Cách 3):**

In [13]:
class Solution:
    def isBipartite(self, graph: List[List[int]]) -> bool:
        from collections import deque
        n = len(graph)
        color = [0] * n
        for s in range(n):
            if color[s]:
                continue
            color[s] = 1
            q = deque([s])
            while q:
                u = q.popleft()
                for v in graph[u]:
                    if color[v] == 0:
                        color[v] = -color[u]
                        q.append(v)
                    elif color[v] == color[u]:
                        return False
        return True

sol_bfs = Solution().isBipartite
print([sol_bfs(g) for g, _ in examples])

[False, True]


**Độ phức tạp (Cách 3):** thời gian `O(n + E)`, bộ nhớ `O(n)` (màu + hàng đợi).

---

### 7.3. Cách 4 — Union Find: mọi láng giềng chung một tập

**Ý tưởng khác.** Trong đồ thị hai phía, **mọi láng giềng của một đỉnh `a` đều nằm ở phía đối diện `a`**, nên chúng **cùng phía với nhau**. Dùng Union Find:

- Với mỗi đỉnh `a` và các láng giềng `bs = graph[a]`: nếu **bất kỳ** `b` nào đã cùng tập với `a` -> xung đột (`a` và `b` kề nhau mà cùng phía).
- Gộp mọi `b` trong `bs` vào cùng một tập (gộp với `bs[0]`).
- Không xung đột nào xảy ra -> hai phía.

**Giải thích từng đoạn.**

- `p[x]` là cha của `x`, `find` có nén đường đi.
- Với mỗi cạnh `(a, b)` duyệt: `pa, pb = find(a), find(b)`; nếu `pa == pb` -> `False`; nếu không `p[pb] = find(bs[0])` (gộp `b` vào tập của láng giềng đầu tiên).

In [14]:
def bip_dsu_neighbors(graph: List[List[int]]) -> bool:
    n = len(graph)
    p = list(range(n))

    def find(x: int) -> int:
        if p[x] != x:
            p[x] = find(p[x])
        return p[x]

    for a, bs in enumerate(graph):
        for b in bs:
            pa, pb = find(a), find(b)
            if pa == pb:
                return False
            p[pb] = find(bs[0])
    return True

for g, expected in examples:
    assert bip_dsu_neighbors(g) == expected
print("Union Find (láng giềng chung tập) OK")

Union Find (láng giềng chung tập) OK


**Code nộp LeetCode (Cách 4):**

In [15]:
class Solution:
    def isBipartite(self, graph: List[List[int]]) -> bool:
        def find(x: int) -> int:
            if p[x] != x:
                p[x] = find(p[x])
            return p[x]

        p = list(range(len(graph)))
        for a, bs in enumerate(graph):
            for b in bs:
                pa, pb = find(a), find(b)
                if pa == pb:
                    return False
                p[pb] = find(bs[0])
        return True

sol_dsu = Solution().isBipartite
print([sol_dsu(g) for g, _ in examples])

[False, True]


**Độ phức tạp (Cách 4):** thời gian `O((n + E) * alpha(n))` (gần tuyến tính nhờ nén đường đi), bộ nhớ `O(n)`.

---

### 7.4. Cách 5 — Union Find có tính chẵn lẻ (weighted DSU)

**Ý tưởng khác.** Mỗi đỉnh lưu thêm `par[x]` = **độ lệch màu** của `x` so với cha `p[x]` (0 cùng màu, 1 khác màu). `find(x)` trả `(gốc, độ lệch của x so với gốc)`. Mỗi cạnh `(a, b)` **ép** `a`, `b` khác màu:

- Nếu `a`, `b` đã cùng gốc: cần độ lệch của chúng khác nhau; nếu bằng nhau -> **xung đột** (chu trình lẻ).
- Nếu khác gốc: gộp hai cây sao cho độ lệch khớp: đặt `p[ra] = rb` và `par[ra] = pa ^ pb ^ 1`.

**Giải thích từng đoạn.**

- `find` đệ quy: trả về `(gốc, độ lệch so với gốc)`; khi nén đường đi, cập nhật `par[x] ^= độ lệch của cha`.
- Phép `^` (XOR) trên 0/1 là cộng mod 2.
- Cách này không cần duyệt theo đồ thị; xử lý **từng cạnh** nên cũng dùng được khi cạnh đến theo dòng (online).

In [16]:
def bip_dsu_parity(graph: List[List[int]]) -> bool:
    n = len(graph)
    p = list(range(n))
    par = [0] * n                     # par[x]: độ lệch màu của x so với p[x]

    def find(x: int):
        if p[x] == x:
            return x, 0
        root, up = find(p[x])         # up: độ lệch của p[x] so với gốc
        par[x] ^= up                  # độ lệch của x so với gốc
        p[x] = root
        return root, par[x]

    for a, bs in enumerate(graph):
        for b in bs:
            if a < b:                 # mỗi cạnh một lần
                ra, pa = find(a)
                rb, pb = find(b)
                if ra == rb:
                    if pa == pb:
                        return False
                else:
                    p[ra] = rb
                    par[ra] = pa ^ pb ^ 1
    return True

for g, expected in examples:
    assert bip_dsu_parity(g) == expected
print("Union Find chẵn lẻ OK")

Union Find chẵn lẻ OK


**Code nộp LeetCode (Cách 5):**

In [17]:
class Solution:
    def isBipartite(self, graph: List[List[int]]) -> bool:
        n = len(graph)
        p = list(range(n))
        par = [0] * n

        def find(x: int):
            if p[x] == x:
                return x, 0
            root, up = find(p[x])
            par[x] ^= up
            p[x] = root
            return root, par[x]

        for a, bs in enumerate(graph):
            for b in bs:
                if a < b:
                    ra, pa = find(a)
                    rb, pb = find(b)
                    if ra == rb:
                        if pa == pb:
                            return False
                    else:
                        p[ra] = rb
                        par[ra] = pa ^ pb ^ 1
        return True

sol_parity = Solution().isBipartite
print([sol_parity(g) for g, _ in examples])

[False, True]


**Độ phức tạp (Cách 5):** thời gian `O((n + E) * alpha(n))` (nén đường đi, không dùng hợp theo hạng nên thực tế vẫn rất nhanh), bộ nhớ `O(n)`.

## 8. TRACE từng bước

### 8.1. DFS trên ví dụ 2 `[[1,3],[0,2],[1,3],[0,2]]`

| bước | lời gọi | hành động |
|---|---|---|
| 1 | `dfs(0, 1)` | tô `0` màu 1; láng giềng 1 chưa tô -> `dfs(1, -1)` |
| 2 | `dfs(1, -1)` | tô `1` màu -1; láng giềng 0 (màu 1 ≠ -1 ✓), láng giềng 2 chưa tô -> `dfs(2, 1)` |
| 3 | `dfs(2, 1)` | tô `2` màu 1; láng giềng 1 (-1 ✓), láng giềng 3 chưa tô -> `dfs(3, -1)` |
| 4 | `dfs(3, -1)` | tô `3` màu -1; láng giềng 0 (1 ✓), láng giềng 2 (1 ✓) |

Kết thúc không xung đột -> `true`.

In [18]:
def trace_dfs(graph):
    n = len(graph)
    color = [0] * n

    def dfs(u, c, depth):
        pad = "  " * depth
        color[u] = c
        print(f"{pad}dfs({u}, {c:2}): tô {u} màu {c:2}")
        for v in graph[u]:
            if color[v] == c:
                print(f"{pad}  láng giềng {v} đã có màu {c} == màu của {u} -> XUNG ĐỘT (cạnh {u}-{v})")
                return False
            if color[v] == 0:
                print(f"{pad}  láng giềng {v} chưa tô -> dfs({v}, {-c})")
                if not dfs(v, -c, depth + 1):
                    return False
            else:
                print(f"{pad}  láng giềng {v} màu {color[v]} khác {c} -> hợp lệ")
        return True

    for i in range(n):
        if color[i] == 0:
            print(f"bắt đầu thành phần mới từ đỉnh {i}")
            if not dfs(i, 1, 0):
                print("kết quả: False")
                return False
    print("kết quả: True, màu =", color)
    return True

assert trace_dfs(examples[1][0]) is True
print()
assert trace_dfs(examples[0][0]) is False

bắt đầu thành phần mới từ đỉnh 0
dfs(0,  1): tô 0 màu  1
  láng giềng 1 chưa tô -> dfs(1, -1)
  dfs(1, -1): tô 1 màu -1
    láng giềng 0 màu 1 khác -1 -> hợp lệ
    láng giềng 2 chưa tô -> dfs(2, 1)
    dfs(2,  1): tô 2 màu  1
      láng giềng 1 màu -1 khác 1 -> hợp lệ
      láng giềng 3 chưa tô -> dfs(3, -1)
      dfs(3, -1): tô 3 màu -1
        láng giềng 0 màu 1 khác -1 -> hợp lệ
        láng giềng 2 màu 1 khác -1 -> hợp lệ
  láng giềng 3 màu -1 khác 1 -> hợp lệ
kết quả: True, màu = [1, -1, 1, -1]

bắt đầu thành phần mới từ đỉnh 0
dfs(0,  1): tô 0 màu  1
  láng giềng 1 chưa tô -> dfs(1, -1)
  dfs(1, -1): tô 1 màu -1
    láng giềng 0 màu 1 khác -1 -> hợp lệ
    láng giềng 2 chưa tô -> dfs(2, 1)
    dfs(2,  1): tô 2 màu  1
      láng giềng 0 đã có màu 1 == màu của 2 -> XUNG ĐỘT (cạnh 2-0)
kết quả: False


### 8.2. BFS trên một đồ thị không liên thông

Đồ thị: thành phần `{0,1,2}` là đường `0-1-2`, thành phần `{3,4,5}` là tam giác (có chu trình lẻ), đỉnh `6` cô lập. BFS từ `0` thành công, rồi từ `3` phát hiện xung đột. Đáp án `False`. Hàm in từng bước và hàng đợi.

In [19]:
def trace_bfs(graph):
    n = len(graph)
    color = [0] * n
    for s in range(n):
        if color[s]:
            continue
        color[s] = 1
        q = deque([s])
        print(f"=== thành phần mới, bắt đầu từ {s} (màu 1) ===")
        while q:
            u = q.popleft()
            print(f"lấy {u} (màu {color[u]:2}); hàng đợi còn {list(q)}")
            for v in graph[u]:
                if color[v] == 0:
                    color[v] = -color[u]
                    q.append(v)
                    print(f"   tô {v} màu {color[v]:2} và đưa vào hàng đợi")
                elif color[v] == color[u]:
                    print(f"   XUNG ĐỘT: {u} và {v} đều màu {color[u]} -> không phải hai phía")
                    return False
                else:
                    print(f"   {v} đã tô màu {color[v]} khác -> hợp lệ")
    print("không xung đột; màu =", color)
    return True

g = build_graph(7, [(0, 1), (1, 2), (3, 4), (4, 5), (5, 3)])
assert trace_bfs(g) is False
print()
g2 = build_graph(7, [(0, 1), (1, 2), (3, 4), (4, 5), (5, 6)])
assert trace_bfs(g2) is True

=== thành phần mới, bắt đầu từ 0 (màu 1) ===
lấy 0 (màu  1); hàng đợi còn []
   tô 1 màu -1 và đưa vào hàng đợi
lấy 1 (màu -1); hàng đợi còn []
   0 đã tô màu 1 khác -> hợp lệ
   tô 2 màu  1 và đưa vào hàng đợi
lấy 2 (màu  1); hàng đợi còn []
   1 đã tô màu -1 khác -> hợp lệ
=== thành phần mới, bắt đầu từ 3 (màu 1) ===
lấy 3 (màu  1); hàng đợi còn []
   tô 4 màu -1 và đưa vào hàng đợi
   tô 5 màu -1 và đưa vào hàng đợi
lấy 4 (màu -1); hàng đợi còn [5]
   3 đã tô màu 1 khác -> hợp lệ
   XUNG ĐỘT: 4 và 5 đều màu -1 -> không phải hai phía

=== thành phần mới, bắt đầu từ 0 (màu 1) ===
lấy 0 (màu  1); hàng đợi còn []
   tô 1 màu -1 và đưa vào hàng đợi
lấy 1 (màu -1); hàng đợi còn []
   0 đã tô màu 1 khác -> hợp lệ
   tô 2 màu  1 và đưa vào hàng đợi
lấy 2 (màu  1); hàng đợi còn []
   1 đã tô màu -1 khác -> hợp lệ
=== thành phần mới, bắt đầu từ 3 (màu 1) ===
lấy 3 (màu  1); hàng đợi còn []
   tô 4 màu -1 và đưa vào hàng đợi
lấy 4 (màu -1); hàng đợi còn []
   3 đã tô màu 1 khác -> hợp lệ
   tô

### 8.3. Union Find chẵn lẻ trên tam giác `0-1-2`

Cạnh `0-1`: gốc khác nhau -> gộp (`p[0] = 1`, `par[0] = 0 ^ 0 ^ 1 = 1`, 0 và 1 khác màu). Cạnh `0-2`: `find(0)` = (1, 1), `find(2)` = (2, 0); gốc khác nhau -> gộp (`p[1] = 2`, `par[1] = 1 ^ 0 ^ 1 = 0`). Cạnh `1-2`: `find(1)` = (2, 0), `find(2)` = (2, 0): **cùng gốc, cùng độ lệch** -> xung đột.

In [20]:
def trace_dsu_parity(n, edges):
    p = list(range(n))
    par = [0] * n

    def find(x):
        if p[x] == x:
            return x, 0
        root, up = find(p[x])
        par[x] ^= up
        p[x] = root
        return root, par[x]

    for a, b in edges:
        ra, pa = find(a)
        rb, pb = find(b)
        print(f"cạnh {a}-{b}: find({a}) = (gốc {ra}, lệch {pa}), find({b}) = (gốc {rb}, lệch {pb})", end=" -> ")
        if ra == rb:
            if pa == pb:
                print("cùng gốc và cùng độ lệch: XUNG ĐỘT")
                return False
            print("cùng gốc, độ lệch khác: hợp lệ")
        else:
            p[ra] = rb
            par[ra] = pa ^ pb ^ 1
            print(f"gộp: p[{ra}] = {rb}, par[{ra}] = {par[ra]}")
    print("không xung đột")
    return True

assert trace_dsu_parity(3, [(0, 1), (0, 2), (1, 2)]) is False
print()
assert trace_dsu_parity(4, [(0, 1), (1, 2), (2, 3), (3, 0)]) is True

cạnh 0-1: find(0) = (gốc 0, lệch 0), find(1) = (gốc 1, lệch 0) -> gộp: p[0] = 1, par[0] = 1
cạnh 0-2: find(0) = (gốc 1, lệch 1), find(2) = (gốc 2, lệch 0) -> gộp: p[1] = 2, par[1] = 0
cạnh 1-2: find(1) = (gốc 2, lệch 0), find(2) = (gốc 2, lệch 0) -> cùng gốc và cùng độ lệch: XUNG ĐỘT

cạnh 0-1: find(0) = (gốc 0, lệch 0), find(1) = (gốc 1, lệch 0) -> gộp: p[0] = 1, par[0] = 1
cạnh 1-2: find(1) = (gốc 1, lệch 0), find(2) = (gốc 2, lệch 0) -> gộp: p[1] = 2, par[1] = 1
cạnh 2-3: find(2) = (gốc 2, lệch 0), find(3) = (gốc 3, lệch 0) -> gộp: p[2] = 3, par[2] = 1
cạnh 3-0: find(3) = (gốc 3, lệch 0), find(0) = (gốc 3, lệch 1) -> cùng gốc, độ lệch khác: hợp lệ
không xung đột


## 9. Code nộp LeetCode

Ta chọn **Cách 2** (DFS tô màu): ngắn, đúng với ghi chú "mọi thành phần liên thông". BFS (Cách 3) tương đương và không đệ quy; Union Find là phương án khác.

```python
class Solution:
    def isBipartite(self, graph: List[List[int]]) -> bool:
        def dfs(a: int, c: int) -> bool:
            color[a] = c
            for b in graph[a]:
                if color[b] == c or (color[b] == 0 and not dfs(b, -c)):
                    return False
            return True

        n = len(graph)
        color = [0] * n
        for i in range(n):
            if color[i] == 0 and not dfs(i, 1):
                return False
        return True
```

In [21]:
class Solution:
    def isBipartite(self, graph: List[List[int]]) -> bool:
        def dfs(a: int, c: int) -> bool:
            color[a] = c
            for b in graph[a]:
                if color[b] == c or (color[b] == 0 and not dfs(b, -c)):
                    return False
            return True

        n = len(graph)
        color = [0] * n
        for i in range(n):
            if color[i] == 0 and not dfs(i, 1):
                return False
        return True

sol = Solution()
for g, expected in examples:
    got = sol.isBipartite(g)
    print(g, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[[1, 2, 3], [0, 2], [0, 1, 3], [0, 2]] -> False OK
[[1, 3], [0, 2], [1, 3], [0, 2]] -> True OK


## 10. Vì sao đúng

**(a) Màu bị ép trong một thành phần.** Nếu đồ thị hai phía có một cách tô hợp lệ, thì với mỗi cạnh `u - v`, `color[v] = -color[u]`. Từ một đỉnh gốc `s` của thành phần, mọi đỉnh `x` có đường đi `s = x_0 - x_1 - ... - x_m = x`, nên `color[x] = (-1)^m * color[s]`: màu của `x` được xác định hoàn toàn bởi `color[s]` và **độ dài (chẵn/lẻ) của một đường đi bất kỳ** từ `s` tới `x`. Nếu hai đường đi tới `x` có độ dài khác tính chẵn lẻ thì hợp chúng thành một **bước đi khép kín độ dài lẻ**, bất khả thi. Vậy chỉ có **đúng hai** cách tô mỗi thành phần (hoán đổi màu), và thuật toán tô màu bằng DFS/BFS thử đúng một trong hai cách; nếu có cách tô hợp lệ thì thuật toán **không bao giờ** thấy xung đột (nó tái tạo đúng cách tô duy nhất đó).

**(b) Xung đột ⇒ không hai phía; không xung đột ⇒ hợp lệ.** Xung đột nghĩa là có cạnh `u - v` với `color[u] == color[v]`, trong khi theo (a) mọi cách tô hợp lệ buộc `color[v] = -color[u]` -> không có cách tô nào -> không hai phía. Ngược lại nếu duyệt xong không xung đột thì mọi cạnh đã được xét (mỗi cạnh được xét ở lần duyệt một trong hai đầu mút của nó) và khác màu: cách tô tìm được hợp lệ. Vòng ngoài phủ mọi thành phần nên mọi cạnh đều được xét.

**(c) Định lý chu trình lẻ.** (⇒) Trong đồ thị hai phía, mọi chu trình đổi phía sau mỗi cạnh nên có độ dài **chẵn**. (⇐) Nếu BFS từ gốc `s` gặp cạnh `u - v` với `color[u] == color[v]` thì `dist(u)` và `dist(v)` cùng chẵn lẻ (màu = chẵn lẻ của khoảng cách); ghép đường đi cây `u -> LCA -> v` với cạnh `u - v` cho một chu trình có độ dài `dist(u) + dist(v) - 2 dist(LCA) + 1` là **số lẻ**. Hàm `find_odd_cycle` dưới đây dựng chu trình này; ta kiểm tra bằng máy rằng nó hợp lệ.

**(d) Cách 4 (Union Find láng giềng).** Trong đồ thị hai phía mọi láng giềng của `a` cùng phía (đối diện `a`), nên gộp chúng không gây mâu thuẫn; và nếu một láng giềng `b` của `a` bị gộp vào cùng tập với `a` thì suy ra có đường đi độ dài chẵn nối `a` và `b` (qua các láng giềng chung) cộng cạnh `a - b` là chu trình lẻ. Tính đúng đắn đầy đủ tương đương (c).

**(e) Cách 5 (Union Find chẵn lẻ).** Bất biến: `par[x]` thuộc `{0, 1}` là độ lệch màu của `x` so với gốc trong **mọi** cách tô hợp lệ của các cạnh đã xét. Cạnh mới ép `a`, `b` khác màu: nếu đã cùng thành phần thì độ lệch đã xác định, phải khác nhau; nếu khác thành phần thì có thể chọn độ lệch gốc cho khớp (`pa ^ par[ra] ^ pb == 1`).

**Kiểm chứng bằng máy.** Mỗi kết luận có một **bằng chứng** dễ kiểm tra: nếu `True` thì tồn tại **một cách tô hợp lệ** (kiểm tra mọi cạnh); nếu `False` thì tồn tại **một chu trình lẻ** (kiểm tra các cạnh liên tiếp và độ dài lẻ). Ta kiểm tra bằng chứng cho mọi cách trên nhiều đồ thị, và đối chiếu vét cạn `2^n` trên đồ thị nhỏ.

In [22]:
def bipartition(graph: List[List[int]]) -> Optional[Tuple[List[int], List[int]]]:
    """Trả (A, B) nếu hai phía, ngược lại None (dùng BFS)."""
    n = len(graph)
    color = [0] * n
    for s in range(n):
        if color[s]:
            continue
        color[s] = 1
        q = deque([s])
        while q:
            u = q.popleft()
            for v in graph[u]:
                if color[v] == 0:
                    color[v] = -color[u]
                    q.append(v)
                elif color[v] == color[u]:
                    return None
    return [i for i in range(n) if color[i] == 1], [i for i in range(n) if color[i] == -1]

def valid_bipartition(graph, A, B) -> bool:
    n = len(graph)
    if sorted(A + B) != list(range(n)) or set(A) & set(B):
        return False
    side = {x: 0 for x in A}
    side.update({x: 1 for x in B})
    return all(side[u] != side[v] for u in range(n) for v in graph[u])

def find_odd_cycle(graph: List[List[int]]) -> Optional[List[int]]:
    """Nếu đồ thị không hai phía, trả về một chu trình lẻ (danh sách đỉnh theo thứ tự), ngược lại None."""
    n = len(graph)
    depth = [-1] * n
    parent = [-1] * n
    for s in range(n):
        if depth[s] != -1:
            continue
        depth[s] = 0
        q = deque([s])
        while q:
            u = q.popleft()
            for v in graph[u]:
                if depth[v] == -1:
                    depth[v] = depth[u] + 1
                    parent[v] = u
                    q.append(v)
                elif depth[v] % 2 == depth[u] % 2:           # cạnh nối hai đỉnh cùng màu
                    a, b = u, v
                    left, right = [a], [b]
                    while a != b:
                        if depth[a] > depth[b]:
                            a = parent[a]; left.append(a)
                        elif depth[b] > depth[a]:
                            b = parent[b]; right.append(b)
                        else:
                            a, b = parent[a], parent[b]
                            left.append(a); right.append(b)
                    return left + right[-2::-1]
    return None

def valid_odd_cycle(graph, cycle) -> bool:
    k = len(cycle)
    if k % 2 == 0 or k < 3 or len(set(cycle)) != k:
        return False
    adj = [set(a) for a in graph]
    return all(cycle[(i + 1) % k] in adj[cycle[i]] for i in range(k))

for g, expected in examples:
    res = bipartition(g)
    cyc = find_odd_cycle(g)
    print(g, "->", "hai phía" if res else "không", "| phân hai:", res, "| chu trình lẻ:", cyc)
    assert (res is not None) == expected and (cyc is None) == expected
    if res:
        assert valid_bipartition(g, *res)
    else:
        assert valid_odd_cycle(g, cyc)
print("bằng chứng cho ví dụ đề: OK")

[[1, 2, 3], [0, 2], [0, 1, 3], [0, 2]] -> không | phân hai: None | chu trình lẻ: [1, 0, 2]
[[1, 3], [0, 2], [1, 3], [0, 2]] -> hai phía | phân hai: ([0, 2], [1, 3]) | chu trình lẻ: None
bằng chứng cho ví dụ đề: OK


In [23]:
def random_graph(rng, n, p):
    edges = [(u, v) for u in range(n) for v in range(u + 1, n) if rng.random() < p]
    return build_graph(n, edges)

def random_bipartite_graph(rng, n, p):
    side = [rng.randint(0, 1) for _ in range(n)]
    edges = [(u, v) for u in range(n) for v in range(u + 1, n) if side[u] != side[v] and rng.random() < p]
    return build_graph(n, edges)

solutions = {
    "brute": bip_brute,
    "dfs": bip_dfs,
    "bfs": bip_bfs,
    "dsu_neighbors": bip_dsu_neighbors,
    "dsu_parity": bip_dsu_parity,
    "sol_dfs": sol_dfs,
    "sol_bfs": sol_bfs,
    "sol_dsu": sol_dsu,
    "sol_parity": sol_parity,
    "final": Solution().isBipartite,
}

# đối chiếu vét cạn trên MỌI đồ thị n <= 5 (mọi tập cạnh) và đồ thị ngẫu nhiên n <= 10
checked = 0
for n in range(1, 6):
    pairs = [(u, v) for u in range(n) for v in range(u + 1, n)]
    for mask in range(1 << len(pairs)):
        g = build_graph(n, [pairs[i] for i in range(len(pairs)) if mask >> i & 1])
        ref = bip_brute(g)
        for name, fn in solutions.items():
            assert fn(g) == ref, (name, g)
        checked += 1
print("mọi đồ thị n = 1..5 (mọi tập cạnh):", checked, "đồ thị x", len(solutions), "cài đặt khớp oracle")

mọi đồ thị n = 1..5 (mọi tập cạnh): 1099 đồ thị x 10 cài đặt khớp oracle


## 11. Time / Space Complexity

Gọi `n` là số đỉnh (`<= 100`), `E` là số cạnh (`<= n(n-1)/2`).

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi cách tô | `O(2^n * E)` | `O(n)` |
| 2. DFS tô màu | `O(n + E)` | `O(n)` |
| 3. BFS tô màu | `O(n + E)` | `O(n)` |
| 4. Union Find (láng giềng chung tập) | `O((n + E) alpha(n))` | `O(n)` |
| 5. Union Find chẵn lẻ | `O((n + E) alpha(n))` | `O(n)` |

Với `n = 100`, `E <= 4950`: tất cả các cách `O(n + E)` chạy dưới 1 ms. Để thấy khác biệt, đo trên đồ thị lớn hơn nhiều so với ràng buộc (`n = 3000`) — vét cạn dĩ nhiên bỏ qua.

In [24]:
rng = random.Random(785)
big_graphs = []
for _ in range(3):
    big_graphs.append(random_bipartite_graph(rng, 1500, 0.004))          # hai phía, thưa
big_graphs.append(random_graph(rng, 1500, 0.003))                         # ngẫu nhiên: gần như chắc chắn có chu trình lẻ
path = build_graph(2000, [(i, i + 1) for i in range(1999)])                # đường dài 2000 (đệ quy sâu 2000: dùng BFS/DSU)
big_graphs.append(path)

import sys
sys.setrecursionlimit(10000)
ref = None
for name in ["bfs", "dsu_neighbors", "dsu_parity", "dfs"]:
    t0 = time.perf_counter()
    res = [solutions[name](g) for g in big_graphs]
    dt = (time.perf_counter() - t0) * 1000
    if ref is None:
        ref = res
    assert res == ref, name
    print(f"{name:14} {len(big_graphs)} đồ thị lớn: {dt:8.2f} ms")
print("kết quả:", ref)

bfs            5 đồ thị lớn:     2.40 ms
dsu_neighbors  5 đồ thị lớn:     9.15 ms
dsu_parity     5 đồ thị lớn:     5.85 ms
dfs            5 đồ thị lớn:     2.46 ms
kết quả: [True, True, True, False, True]


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai, đối chiếu mọi cách với oracle vét cạn và với **bằng chứng** (cách tô hợp lệ hoặc chu trình lẻ), và test ngẫu nhiên có seed cố định (đúng ràng buộc: đồ thị vô hướng đơn, không khuyên, `n <= 100`).

In [25]:
# 12.1 ví dụ đề
for g, expected in examples:
    for name, fn in solutions.items():
        assert fn(g) == expected, (name, g)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay)
edge = [
    ([[]], True),                                         # một đỉnh, không cạnh
    ([[], []], True),                                     # hai đỉnh rời nhau
    ([[1], [0]], True),                                   # một cạnh
    ([[1, 2], [0, 2], [0, 1]], False),                    # tam giác
    (build_graph(4, [(0, 1), (1, 2), (2, 3), (3, 0)]), True),            # chu trình 4
    (build_graph(5, [(0, 1), (1, 2), (2, 3), (3, 4), (4, 0)]), False),   # chu trình 5
    (build_graph(6, [(i, (i + 1) % 6) for i in range(6)]), True),        # chu trình 6
    (build_graph(7, [(0, 1), (1, 2), (3, 4), (4, 5), (5, 3)]), False),   # thành phần thứ hai là tam giác (không liên thông!)
    (build_graph(7, [(0, 1), (1, 2), (3, 4), (4, 5), (5, 6)]), True),    # hai đường + không liên thông
    (build_graph(4, [(0, 1), (0, 2), (0, 3)]), True),                    # sao
    (build_graph(5, [(i, j) for i in range(5) for j in range(i + 1, 5)]), False),   # đồ thị đầy đủ K5
    (build_graph(5, [(i, j) for i in range(2) for j in range(2, 5)]), True),         # hai phía đầy đủ K(2,3)
    (build_graph(8, [(i, j) for i in range(4) for j in range(4, 8)]), True),         # K(4,4)
    (build_graph(3, [(0, 1)]), True),                                     # đỉnh 2 cô lập
    (build_graph(100, [(i, i + 1) for i in range(99)]), True),            # đường 100 đỉnh
    (build_graph(99, [(i, (i + 1) % 99) for i in range(99)]), False),     # chu trình 99 (lẻ)
    (build_graph(100, [(i, (i + 1) % 100) for i in range(100)]), True),   # chu trình 100 (chẵn)
]
for g, expected in edge:
    for name, fn in solutions.items():
        if name == "brute" and len(g) > 14:
            continue
        assert fn(g) == expected, (name, g, fn(g), expected)
    res, cyc = bipartition(g), find_odd_cycle(g)
    assert (res is not None) == expected and (cyc is None) == expected
    if expected:
        assert valid_bipartition(g, *res)
    else:
        assert valid_odd_cycle(g, cyc), cyc
print("12.2 edge case: OK", len(edge), "ca")

# 12.3 phản ví dụ: các cách sai
def wrong_only_zero(graph):                  # chỉ duyệt từ đỉnh 0 (bỏ các thành phần khác)
    n = len(graph)
    color = [0] * n
    color[0] = 1
    q = deque([0])
    while q:
        u = q.popleft()
        for v in graph[u]:
            if color[v] == 0:
                color[v] = -color[u]
                q.append(v)
            elif color[v] == color[u]:
                return False
    return True

def wrong_01_colors(graph):                  # màu 0/1 trùng với "chưa tô" (0)
    n = len(graph)
    color = [0] * n
    for s in range(n):
        if color[s]:
            continue
        color[s] = 1
        q = deque([s])
        while q:
            u = q.popleft()
            for v in graph[u]:
                if color[v] == 0:
                    color[v] = 1 - color[u]         # đổi 1 <-> 0: 0 lại nghĩa là "chưa tô"
                    q.append(v)
                elif color[v] == color[u]:
                    return False
    return True

def wrong_visited(graph):                    # dùng visited thay vì màu: không bao giờ thấy xung đột
    n = len(graph)
    seen = [False] * n
    for s in range(n):
        if seen[s]:
            continue
        seen[s] = True
        q = deque([s])
        while q:
            u = q.popleft()
            for v in graph[u]:
                if not seen[v]:
                    seen[v] = True
                    q.append(v)
    return True

def wrong_cycle_means_no(graph):             # "có chu trình thì không hai phía"
    n = len(graph)
    edges = sum(len(a) for a in graph) // 2
    comps = len(components(graph))
    return edges == n - comps                 # đồ thị là rừng (không có chu trình)

def wrong_no_neighbor_check(graph):          # DFS không so với màu của láng giềng đã tô
    n = len(graph)
    color = [0] * n
    def dfs(u, c):
        color[u] = c
        for v in graph[u]:
            if color[v] == 0 and not dfs(v, -c):
                return False
        return True
    return all(color[i] != 0 or dfs(i, 1) for i in range(n))

wrongs = [
    ("chỉ duyệt từ đỉnh 0", wrong_only_zero, build_graph(7, [(0, 1), (1, 2), (3, 4), (4, 5), (5, 3)])),
    ("màu 0/1 trùng chưa tô", wrong_01_colors, [[1], [0]]),
    ("visited thay vì màu", wrong_visited, [[1, 2], [0, 2], [0, 1]]),
    ("có chu trình => không hai phía", wrong_cycle_means_no, build_graph(4, [(0, 1), (1, 2), (2, 3), (3, 0)])),
    ("không so màu láng giềng đã tô", wrong_no_neighbor_check, [[1, 2], [0, 2], [0, 1]]),
]
for name, fn, g in wrongs:
    truth = bip_brute(g)
    print(f"{name:32} đồ thị {len(g)} đỉnh: sai = {fn(g)}, đúng = {truth}")
    assert fn(g) != truth, name
print("12.3 phản ví dụ: OK (các cách sai đều bị lộ)")

12.1 ví dụ đề: OK
12.2 edge case: OK 17 ca
chỉ duyệt từ đỉnh 0              đồ thị 7 đỉnh: sai = True, đúng = False
màu 0/1 trùng chưa tô            đồ thị 2 đỉnh: sai = False, đúng = True
visited thay vì màu              đồ thị 3 đỉnh: sai = True, đúng = False
có chu trình => không hai phía   đồ thị 4 đỉnh: sai = False, đúng = True
không so màu láng giềng đã tô    đồ thị 3 đỉnh: sai = True, đúng = False
12.3 phản ví dụ: OK (các cách sai đều bị lộ)


In [26]:
# 12.4 ngẫu nhiên nhỏ so với oracle vét cạn (n <= 11)
rng = random.Random(2785)
n_true = 0
for _ in range(500):
    n = rng.randint(1, 11)
    g = random_bipartite_graph(rng, n, rng.choice([0.2, 0.5, 0.9])) if rng.random() < 0.5 else random_graph(rng, n, rng.choice([0.1, 0.2, 0.4]))
    ref = bip_brute(g)
    for name, fn in solutions.items():
        assert fn(g) == ref, (name, g)
    n_true += ref
print(f"12.4 ngẫu nhiên n <= 11 so với vét cạn: OK ({n_true} hai phía / {500 - n_true} không)")

# 12.5 ngẫu nhiên lớn tới n = 100: đối chiếu bằng BẰNG CHỨNG (cách tô hợp lệ hoặc chu trình lẻ hợp lệ)
rng = random.Random(77)
n_true = 0
for _ in range(400):
    n = rng.randint(20, 100)
    kind = rng.random()
    if kind < 0.4:
        g = random_bipartite_graph(rng, n, rng.choice([0.02, 0.05, 0.3]))
    elif kind < 0.7:
        g = random_graph(rng, n, rng.choice([0.01, 0.02, 0.05]))
    else:
        # hai phía + đúng một cạnh nội bộ thêm vào (có thể tạo chu trình lẻ, có thể không nếu khác thành phần)
        g = random_bipartite_graph(rng, n, 0.05)
        u, v = rng.sample(range(n), 2)
        if v not in g[u]:
            g[u].append(v); g[v].append(u)
    res = {name: fn(g) for name, fn in solutions.items() if name != "brute"}
    assert len(set(res.values())) == 1, (g, res)
    answer = res["final"]
    n_true += answer
    part = bipartition(g)
    cyc = find_odd_cycle(g)
    assert (part is not None) == answer and (cyc is None) == answer
    if answer:
        assert valid_bipartition(g, *part)
    else:
        assert valid_odd_cycle(g, cyc), cyc
print(f"12.5 ngẫu nhiên n tới 100 đối chiếu bằng chứng: OK ({n_true} hai phía / {400 - n_true} không)")

# 12.6 đồ thị cực đại theo ràng buộc: n = 100
n = 100
K_half = build_graph(n, [(i, j) for i in range(50) for j in range(50, 100)])           # K(50,50): 2500 cạnh
assert all(fn(K_half) for name, fn in solutions.items() if name != "brute")
K_full = build_graph(n, [(i, j) for i in range(n) for j in range(i + 1, n)])          # K100: 4950 cạnh
assert not any(fn(K_full) for name, fn in solutions.items() if name != "brute")
assert valid_odd_cycle(K_full, find_odd_cycle(K_full))
print("12.6 n = 100: K(50,50) hai phía, K100 không: OK")

12.4 ngẫu nhiên n <= 11 so với vét cạn: OK (423 hai phía / 77 không)


12.5 ngẫu nhiên n tới 100 đối chiếu bằng chứng: OK (314 hai phía / 86 không)
12.6 n = 100: K(50,50) hai phía, K100 không: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ duyệt từ đỉnh 0

Đồ thị có thể **không liên thông**: thành phần không chứa đỉnh 0 bị bỏ qua, mọi chu trình lẻ ở đó bị sót.

In [27]:
g = build_graph(7, [(0, 1), (1, 2), (3, 4), (4, 5), (5, 3)])
print("sai (chỉ từ đỉnh 0):", wrong_only_zero(g), "| đúng:", Solution().isBipartite(g))
assert wrong_only_zero(g) is True and Solution().isBipartite(g) is False

sai (chỉ từ đỉnh 0): True | đúng: False


Vì sao sai: tam giác `3-4-5` nằm ở thành phần khác. **Đúng:** vòng ngoài `for i in range(n): if color[i] == 0: ...` để phủ mọi thành phần.

### Lỗi 2 — Dùng màu `0/1` trong khi `0` cũng nghĩa là "chưa tô"

Nếu `color` khởi tạo bằng `0` (chưa tô) mà lại dùng `0` làm một trong hai màu, ta không phân biệt được "đã tô màu 0" với "chưa tô".

In [28]:
g = [[1], [0]]                       # chỉ một cạnh: hiển nhiên là hai phía
print("sai (màu 0/1):", wrong_01_colors(g), "| đúng:", Solution().isBipartite(g))
assert wrong_01_colors(g) is False and Solution().isBipartite(g) is True

sai (màu 0/1): False | đúng: True


Vì sao sai: đỉnh `1` được tô màu `0` bị coi là **chưa tô**, nên vòng ngoài bắt đầu lại từ nó với màu 1 và va chạm với đỉnh `0` (cũng màu 1): báo sai là không hai phía dù chỉ có một cạnh. **Đúng:** dùng `0` = chưa tô, `1` và `-1` là hai màu (hoặc mảng riêng `colored`).

### Lỗi 3 — Dùng `visited` thay vì màu

`visited` chỉ nói "đã thăm", không cho biết đỉnh **cùng phía hay khác phía**, nên không thể phát hiện cạnh nối hai đỉnh cùng màu.

In [29]:
g = [[1, 2], [0, 2], [0, 1]]
print("sai (visited):", wrong_visited(g), "| đúng:", Solution().isBipartite(g))
assert wrong_visited(g) is True and Solution().isBipartite(g) is False

sai (visited): True | đúng: False


Vì sao sai: duyệt thành công mọi đỉnh nhưng tam giác vẫn tồn tại. **Đúng:** lưu màu, và so màu khi gặp đỉnh đã tô.

### Lỗi 4 — "Có chu trình thì không hai phía"

Chỉ chu trình **lẻ** mới cản trở. Chu trình chẵn (như hình vuông 4 đỉnh) vẫn là hai phía.

In [30]:
g = build_graph(4, [(0, 1), (1, 2), (2, 3), (3, 0)])
print("sai (có chu trình => không):", wrong_cycle_means_no(g), "| đúng:", Solution().isBipartite(g))
assert wrong_cycle_means_no(g) is False and Solution().isBipartite(g) is True

sai (có chu trình => không): False | đúng: True


Vì sao sai: chu trình 4 đổi phía sau mỗi cạnh và quay lại đúng phía cũ (4 là số chẵn). **Đúng:** kiểm tra tính chẵn lẻ của chu trình, tức tô màu.

### Lỗi 5 — Không so với màu của láng giềng đã tô

Chỉ đệ quy vào láng giềng **chưa tô** mà bỏ qua việc kiểm tra láng giềng **đã tô** thì xung đột bị bỏ sót (xuất hiện ở "cạnh ngược" của DFS).

In [31]:
g = [[1, 2], [0, 2], [0, 1]]
print("sai (không so màu):", wrong_no_neighbor_check(g), "| đúng:", Solution().isBipartite(g))
assert wrong_no_neighbor_check(g) is True and Solution().isBipartite(g) is False

sai (không so màu): True | đúng: False


Vì sao sai: cạnh `1 - 2` nối hai đỉnh đã tô cùng màu nhưng không ai kiểm tra. **Đúng:** `if color[b] == c: return False` cho mọi láng giềng đã tô.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "chia thành hai nhóm sao cho các cặp xung đột thuộc nhóm khác nhau", "hai phe", "chia đội" -> **đồ thị hai phía** (tô hai màu).
- **Định lý:** hai phía ⇔ không có chu trình lẻ. Ghi nhớ để phân biệt với "không có chu trình".
- **Luôn bao phủ mọi thành phần liên thông** bằng vòng ngoài.
- **BFS/DFS/Union Find** đều đạt `O(n + E)`; chọn BFS khi muốn tránh đệ quy sâu (ở bài này `n <= 100` nên đệ quy an toàn).
- **Màu bị ép:** trong mỗi thành phần chỉ có đúng 2 cách tô; vì vậy số cách tô hợp lệ = `2^(số thành phần)` nếu hai phía (bài tập mở rộng).
- Khi cần **bằng chứng**: cách tô (nếu có) hoặc chu trình lẻ (nếu không).
- Biến thể trực tiếp: Possible Bipartition (886), Divide Nodes Into the Maximum Number of Groups (2493).
- Test nên có: đồ thị không liên thông, đỉnh cô lập, chu trình chẵn/lẻ, đồ thị đầy đủ hai phía, `K5`.

## 15. Biến thể

1. **Trả về hai tập** `(A, B)` (đã có `bipartition`).
2. **Đếm số cách chia** thành hai tập (có thứ tự `A`/`B`): `2^c` với `c` là số thành phần liên thông nếu hai phía, ngược lại `0`.
3. **Possible Bipartition (886):** `n` người (đánh số từ 1), danh sách các cặp ghét nhau; chia hai nhóm sao cho không cặp ghét nào cùng nhóm. Dựng đồ thị từ danh sách cặp rồi tô màu.
4. **Chu trình lẻ ngắn nhất** (girth lẻ) bằng BFS từ mọi đỉnh.
5. **Số cạnh tối thiểu cần xóa** để thành hai phía (Max-Cut): NP-khó.

Dưới đây là code cho biến thể 2 và 3, kiểm bằng oracle.

In [32]:
def count_bipartitions(graph) -> int:
    if bipartition(graph) is None:
        return 0
    return 2 ** len(components(graph))

def count_oracle(graph) -> int:
    n = len(graph)
    edges = edge_list(graph)
    return sum(1 for colors in product((1, -1), repeat=n) if all(colors[u] != colors[v] for u, v in edges))

def possible_bipartition(n: int, dislikes: List[List[int]]) -> bool:
    graph = [[] for _ in range(n)]
    for a, b in dislikes:
        graph[a - 1].append(b - 1)
        graph[b - 1].append(a - 1)
    return bipartition(graph) is not None

assert possible_bipartition(4, [[1, 2], [1, 3], [2, 4]]) is True
assert possible_bipartition(3, [[1, 2], [1, 3], [2, 3]]) is False
assert possible_bipartition(5, [[1, 2], [2, 3], [3, 4], [4, 5], [1, 5]]) is False
rng = random.Random(886)
for _ in range(200):
    n = rng.randint(1, 9)
    g = random_graph(rng, n, rng.choice([0.1, 0.25, 0.5]))
    assert count_bipartitions(g) == count_oracle(g), g
print("số cách chia của đường 5 đỉnh:", count_bipartitions(build_graph(5, [(0, 1), (1, 2), (2, 3), (3, 4)])), "(= 2^1)")
print("Biến thể: OK")

số cách chia của đường 5 đỉnh: 2 (= 2^1)
Biến thể: OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi cách tô | Định nghĩa trực tiếp | `O(2^n E)` | `O(n)` | Oracle, `n` rất nhỏ |
| 2. DFS tô màu | Màu bị ép, đệ quy | `O(n + E)` | `O(n)` | **Cách nộp chính**, ngắn gọn |
| 3. BFS tô màu | Cùng luật, hàng đợi | `O(n + E)` | `O(n)` | Tránh đệ quy sâu; tìm chu trình lẻ ngắn |
| 4. Union Find (láng giềng) | Láng giềng của một đỉnh cùng phía | `O((n+E) alpha)` | `O(n)` | Khi đã dùng Union Find; cạnh đến theo dòng |
| 5. Union Find chẵn lẻ | Độ lệch màu so với gốc | `O((n+E) alpha)` | `O(n)` | Bài toán online / Possible Bipartition mở rộng |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng sau, rồi chạy để kiểm tra:

```python
print(build_graph(3, [(0, 1), (1, 2), (2, 0)]))
print(components(build_graph(5, [(0, 1), (2, 3)])))
print(Solution().isBipartite(build_graph(5, [(0, 1), (1, 2), (2, 3), (3, 4), (4, 0)])))
```

In [33]:
# Bài 1: ghi dự đoán của bạn (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với đồ thị ví dụ 2 `[[1,3],[0,2],[1,3],[0,2]]`, tô màu bằng BFS từ đỉnh 0 (màu `1`): ghi thứ tự đỉnh được lấy ra khỏi hàng đợi, mảng màu cuối, và hai tập `A` (màu 1), `B` (màu -1).

In [34]:
# Bài 2: điền kết quả tính tay
hand_order = None     # thứ tự lấy ra khỏi hàng đợi, ví dụ [0, ...]
hand_color = None     # list 4 phần tử
hand_A = None
hand_B = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành BFS tô màu.

In [35]:
def exercise_3_bfs(graph: List[List[int]]) -> bool:
    n = len(graph)
    color = [0] * n
    for s in range(n):
        # TODO 1: bỏ qua đỉnh đã tô; ngược lại tô s màu 1 và tạo hàng đợi chứa s
        # TODO 2: lấy một đỉnh u ra khỏi hàng đợi
        # TODO 3: với mỗi láng giềng v: nếu chưa tô thì tô -color[u] và đưa vào hàng đợi;
        #         nếu đã tô cùng màu với u thì trả False
        pass
    return None


# assert exercise_3_bfs([[1, 3], [0, 2], [1, 3], [0, 2]]) is True
# assert exercise_3_bfs([[1, 2, 3], [0, 2], [0, 1, 3], [0, 2]]) is False
# assert exercise_3_bfs([[], []]) is True

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 2, hãy tự viết `exercise_4_iterative_dfs(graph)` dùng DFS **không đệ quy** (ngăn xếp bằng `list`) để tô màu và kiểm tra hai phía.

In [36]:
def exercise_4_iterative_dfs(graph: List[List[int]]) -> bool:
    # TODO
    pass


# assert exercise_4_iterative_dfs([[1, 3], [0, 2], [1, 3], [0, 2]]) is True
# assert exercise_4_iterative_dfs([[1, 2, 3], [0, 2], [0, 1, 3], [0, 2]]) is False
# assert exercise_4_iterative_dfs(build_graph(100, [(i, i + 1) for i in range(99)])) is True

### Bài 5 — Mini challenge: Possible Bipartition (khó)

`n` người (đánh số `1..n`) và danh sách `dislikes`, mỗi phần tử `[a, b]` nghĩa là `a` và `b` ghét nhau. Trả về `True` nếu có thể chia mọi người vào **hai nhóm** sao cho **không cặp ghét nhau nào ở cùng nhóm**.

```python
exercise_5_possible_bipartition(4, [[1,2],[1,3],[2,4]])          -> True
exercise_5_possible_bipartition(3, [[1,2],[1,3],[2,3]])          -> False
exercise_5_possible_bipartition(5, [[1,2],[2,3],[3,4],[4,5],[1,5]]) -> False
```
Gợi ý: đổi chỉ số về 0, dựng danh sách kề, rồi tô màu như bài này (đừng quên người không có cặp ghét nào).

In [37]:
def exercise_5_possible_bipartition(n: int, dislikes: List[List[int]]) -> bool:
    # TODO
    pass


# assert exercise_5_possible_bipartition(4, [[1, 2], [1, 3], [2, 4]]) is True
# assert exercise_5_possible_bipartition(3, [[1, 2], [1, 3], [2, 3]]) is False
# assert exercise_5_possible_bipartition(5, [[1, 2], [2, 3], [3, 4], [4, 5], [1, 5]]) is False

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `build_graph` thêm cạnh vào cả hai danh sách kề.
- Mức 2: tam giác: `0` kề `1, 2`; `components` của hai cạnh rời + một đỉnh cô lập có 3 thành phần; chu trình 5 là lẻ.
- Mức 3: `[[1, 2], [0, 2], [1, 0]]`... (thứ tự phần tử trong danh sách kề theo thứ tự cạnh được thêm: kiểm tra bằng code); `[[0, 1], [2, 3], [4]]`; `False`.

**Bài 2**
- Mức 1: bắt đầu hàng đợi `[0]`, `color[0] = 1`.
- Mức 2: lấy `0`: tô `1` và `3` màu -1; lấy `1`: tô `2` màu 1; lấy `3`; lấy `2`.
- Mức 3: thứ tự `[0, 1, 3, 2]`, màu `[1, -1, 1, -1]`, `A = [0, 2]`, `B = [1, 3]`.

**Bài 3**
- Mức 1: `if color[s]: continue`; `color[s] = 1`; `q = deque([s])`.
- Mức 2: `while q: u = q.popleft()`.
- Mức 3: `for v in graph[u]: if color[v] == 0: color[v] = -color[u]; q.append(v)`, `elif color[v] == color[u]: return False`; cuối `return True`.

**Bài 4**
- Mức 1: ngăn xếp `stack = [s]` thay cho đệ quy; tô `s` màu 1.
- Mức 2: `u = stack.pop()`; với mỗi láng giềng `v`: chưa tô thì tô `-color[u]` và `push`; đã tô cùng màu thì `return False`.
- Mức 3: vòng ngoài phủ mọi thành phần; cuối `return True`.

**Bài 5**
- Mức 1: `graph = [[] for _ in range(n)]`; với `[a, b]` thêm `b - 1` vào `graph[a - 1]` và ngược lại.
- Mức 2: dùng bất kỳ cách tô màu nào (BFS) ở trên.
- Mức 3: người không có cặp ghét nào có danh sách kề rỗng và luôn thỏa.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [38]:
print(build_graph(3, [(0, 1), (1, 2), (2, 0)]))
print(components(build_graph(5, [(0, 1), (2, 3)])))
print(Solution().isBipartite(build_graph(5, [(0, 1), (1, 2), (2, 3), (3, 4), (4, 0)])))

assert build_graph(3, [(0, 1), (1, 2), (2, 0)]) == [[1, 2], [0, 2], [1, 0]]
assert components(build_graph(5, [(0, 1), (2, 3)])) == [[0, 1], [2, 3], [4]]
assert Solution().isBipartite(build_graph(5, [(0, 1), (1, 2), (2, 3), (3, 4), (4, 0)])) is False

[[1, 2], [0, 2], [1, 0]]
[[0, 1], [2, 3], [4]]
False


### Đáp án Bài 2

Hàng đợi: `[0]` -> lấy `0` (tô `1`, `3` màu -1) -> lấy `1` (tô `2` màu 1) -> lấy `3` -> lấy `2`. Thứ tự `[0, 1, 3, 2]`; màu `[1, -1, 1, -1]`; `A = [0, 2]`, `B = [1, 3]`.

In [39]:
g = [[1, 3], [0, 2], [1, 3], [0, 2]]
order, color = [], [0] * 4
color[0] = 1
q = deque([0])
while q:
    u = q.popleft()
    order.append(u)
    for v in g[u]:
        if color[v] == 0:
            color[v] = -color[u]
            q.append(v)
hand_order_solution = [0, 1, 3, 2]
hand_color_solution = [1, -1, 1, -1]
hand_A_solution, hand_B_solution = [0, 2], [1, 3]
assert order == hand_order_solution and color == hand_color_solution
assert valid_bipartition(g, hand_A_solution, hand_B_solution)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [40]:
def exercise_3_bfs_solution(graph: List[List[int]]) -> bool:
    n = len(graph)
    color = [0] * n
    for s in range(n):
        if color[s]:
            continue
        color[s] = 1
        q = deque([s])
        while q:
            u = q.popleft()
            for v in graph[u]:
                if color[v] == 0:
                    color[v] = -color[u]
                    q.append(v)
                elif color[v] == color[u]:
                    return False
    return True

assert exercise_3_bfs_solution([[1, 3], [0, 2], [1, 3], [0, 2]]) is True
assert exercise_3_bfs_solution([[1, 2, 3], [0, 2], [0, 1, 3], [0, 2]]) is False
assert exercise_3_bfs_solution([[], []]) is True
rng = random.Random(3)
for _ in range(200):
    g = random_graph(rng, rng.randint(1, 9), rng.choice([0.15, 0.3, 0.5]))
    assert exercise_3_bfs_solution(g) == bip_brute(g), g
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [41]:
def exercise_4_iterative_dfs_solution(graph: List[List[int]]) -> bool:
    n = len(graph)
    color = [0] * n
    for s in range(n):
        if color[s]:
            continue
        color[s] = 1
        stack = [s]
        while stack:
            u = stack.pop()
            for v in graph[u]:
                if color[v] == 0:
                    color[v] = -color[u]
                    stack.append(v)
                elif color[v] == color[u]:
                    return False
    return True

assert exercise_4_iterative_dfs_solution([[1, 3], [0, 2], [1, 3], [0, 2]]) is True
assert exercise_4_iterative_dfs_solution([[1, 2, 3], [0, 2], [0, 1, 3], [0, 2]]) is False
assert exercise_4_iterative_dfs_solution(build_graph(100, [(i, i + 1) for i in range(99)])) is True
rng = random.Random(4)
for _ in range(200):
    g = random_graph(rng, rng.randint(1, 9), rng.choice([0.15, 0.3, 0.5]))
    assert exercise_4_iterative_dfs_solution(g) == bip_brute(g), g
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [42]:
def exercise_5_possible_bipartition_solution(n: int, dislikes: List[List[int]]) -> bool:
    graph = [[] for _ in range(n)]
    for a, b in dislikes:
        graph[a - 1].append(b - 1)
        graph[b - 1].append(a - 1)
    color = [0] * n
    for s in range(n):
        if color[s]:
            continue
        color[s] = 1
        q = deque([s])
        while q:
            u = q.popleft()
            for v in graph[u]:
                if color[v] == 0:
                    color[v] = -color[u]
                    q.append(v)
                elif color[v] == color[u]:
                    return False
    return True

assert exercise_5_possible_bipartition_solution(4, [[1, 2], [1, 3], [2, 4]]) is True
assert exercise_5_possible_bipartition_solution(3, [[1, 2], [1, 3], [2, 3]]) is False
assert exercise_5_possible_bipartition_solution(5, [[1, 2], [2, 3], [3, 4], [4, 5], [1, 5]]) is False
rng = random.Random(5)
for _ in range(200):
    n = rng.randint(1, 9)
    pairs = [[a, b] for a in range(1, n + 1) for b in range(a + 1, n + 1) if rng.random() < 0.3]
    expected = bip_brute(build_graph(n, [(a - 1, b - 1) for a, b in pairs]))
    assert exercise_5_possible_bipartition_solution(n, pairs) == expected, (n, pairs)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (DFS tô hai màu)

```python
class Solution:
    def isBipartite(self, graph: List[List[int]]) -> bool:
        n = len(graph)
        color = [0] * n                          # 0 = chưa tô, 1 / -1 = hai màu

        def dfs(u: int, c: int) -> bool:
            color[u] = c
            for v in graph[u]:
                if color[v] == c:                # cạnh nối hai đỉnh cùng màu -> chu trình lẻ
                    return False
                if color[v] == 0 and not dfs(v, -c):
                    return False
            return True

        for i in range(n):                       # phủ mọi thành phần liên thông
            if color[i] == 0 and not dfs(i, 1):
                return False
        return True
```

Công thức nhớ nhanh: hai phía ⇔ không có chu trình lẻ ⇔ tô được hai màu sao cho mọi cạnh nối hai màu khác nhau.

### Checklist trước khi submit

- [ ] Vòng ngoài phủ **mọi** đỉnh chưa tô (đồ thị không liên thông)?
- [ ] `0` = chưa tô, hai màu là `1` và `-1` (không dùng `0/1`)?
- [ ] Khi gặp láng giềng **đã tô**, đã so màu với đỉnh hiện tại?
- [ ] Màu láng giềng chưa tô là `-màu hiện tại`?
- [ ] Không nhầm "có chu trình" với "có chu trình lẻ"?
- [ ] Đã thử đỉnh cô lập, tam giác, chu trình 4 và 5, hai thành phần?
- [ ] Tên method đúng `isBipartite(self, graph)` và trả `bool`?